# 12: Ablations on the frozen model

Design decisions
- One factor at a time; everything else as frozen (model, candidate sets of notebook 08, calibration split 1 for certificates, the explained patients of calibration split 2 for the policy test, targets alpha_0 = 0.05 and alpha_1 = 0.70, delta_1 = delta_2 = 0.025, alpha_sel per task and kind). The frozen configuration is not changed by these runs.
- Factors and levels: prior (Occam only; mixture with lambda 0.25, 0.5, 0.75; training support only), explanation construction (AXps only; widest inflated box of each order only; every sound box on the inflation path), selection rule (most succinct; largest support; tightest bound), number of candidate orders M (1, 3, 5, 10), and size of calibration split 1 (random subsamples of 5,000, 10,000, 20,000 and 40,000 records, 10 draws each, against the full split).
- Results on the 5,000 explained test patients: share explained per kind, test error, explanation size, calibration support; policy test on calibration split 2 at the frozen alpha_sel. Boxes that become releasable under a variant get the second exact check first.
- Development results for M and lambda are the notebook 05 runs on the development splits, listed alongside.
- Model factors (family, depth, trees, monotone constraints, max_bin) and extra model seeds are in notebook 12b.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys, warnings


def quiet_warnings():
    # imodels switches on every DeprecationWarning; Colab's message code then warns on every output line
    warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
    warnings.filterwarnings('ignore', message=r'datetime\.datetime\.utcnow')


quiet_warnings()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, time
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, sha256_file, write_json
from scax.data import public, splits
from scax.experiments.main_results import FrozenTask
try:
    from scax.experiments.baselines import redraw_split, candidate_flags
    from scax.experiments.ablations import FrozenAblation
except ImportError:
    raise RuntimeError('This session still has the old SCAX code loaded. Runtime > Disconnect and delete runtime, then run again.')
from scax.explain.candidates import CandidateGenerator

SEED = 42
seed_everything(SEED)
TASKS = ['cdc_diabetes', 'brfss2015_heart']
NAMES = {'cdc_diabetes': 'CDC Diabetes', 'brfss2015_heart': 'BRFSS Heart disease'}
KIND = {0: 'negative outcome', 1: 'enrichment'}
REC05 = json.load(open(f'{paths.RESULTS}/05_frozen.json'))
REC08 = json.load(open(f'{paths.RESULTS}/08_certificates.json'))
FT = {t: FrozenTask(t, REPO_DIR, DRIVE_ROOT, REC05['frozen_config_sha256'][t]) for t in TASKS}
CERT_DIR = paths.DERIVED / 'certificates'
ABL_DIR = paths.DERIVED / 'ablations'; ABL_DIR.mkdir(parents=True, exist_ok=True)
N1_SIZES, N1_DRAWS = [5000, 10000, 20000, 40000], 10


def load_task(task):
    X, y = public.LOADERS[task](paths.RAW)
    code = np.load(paths.DERIVED / 'splits' / f'{task}.npy')
    assert splits.summary(code)['sha256'] == FT[task].cfg['split_sha256'], 'split differs from the frozen one'
    return X.to_numpy(dtype=float), y, code


def generator(ft):
    c = ft.cfg['candidates']
    gen = CandidateGenerator(ft.pred, M=c['M'], seed=c['order_seed'], node_budget=c['node_budget'])
    assert gen.load(CERT_DIR / f'{ft.task}__candidates.pkl')
    return gen


for t in TASKS:
    assert sha256_file(CERT_DIR / f'{t}__boxes.parquet') == REC08['box_table_sha256'][t], 'box table differs from notebook 08'
    print(t, 'frozen configuration', FT[t].cfg_sha[:16], '| M', FT[t].M, '| lambda', FT[t].lam)

## One factor at a time on the final splits

In [ ]:
ABL = []
for ti, task in enumerate(TASKS):
    f = ABL_DIR / f'{task}__frozen_model.parquet'
    if f.exists():
        ABL.append(pd.read_parquet(f)); print(task, 'loaded from Drive'); continue
    ft = FT[task]
    Xa, y, code = load_task(task)
    t0 = time.time()
    fa = FrozenAblation(ft, pd.read_parquet(CERT_DIR / f'{task}__boxes.parquet'), generator(ft), Xa, y, code,
                        ft.targets['delta_1'], ft.targets['delta_2'])
    status_file = paths.DERIVED / 'baselines' / f'{task}__second_check_status.npy'
    if status_file.exists():
        st = np.load(status_file)
        if len(st) == len(fa.status) and np.all(st[fa.status != 0] == fa.status[fa.status != 0]):
            fa.status = st
    print(task)
    df = fa.run_factors(N1_SIZES, N1_DRAWS, seed=12000 + ti)
    df.insert(0, 'task', task)
    df['source'] = 'final splits'
    df.to_parquet(f, index=False)
    ABL.append(df)
    print(f'{task}: {time.time() - t0:.0f}s')
    del fa
ABL = pd.concat(ABL, ignore_index=True)

# the frozen level must reproduce notebook 09
m09 = pd.read_csv(f'{paths.RESULTS}/09_main.csv')
for task in TASKS:
    for kk, kind in KIND.items():
        a = ABL[(ABL.task == task) & (ABL.factor == 'prior') & ABL.frozen & (ABL.kind == kk)].iloc[0]
        b = m09[(m09.task == task) & m09.frozen_targets & (m09.kind == kind)].iloc[0]
        assert a.accepted == b.test_accepted and a.cal2_accepted == b.cal2_accepted and np.isclose(a.U_sel, b.U_sel), (task, kind)
print('frozen level reproduces notebook 09 for both tasks and both kinds of claim')

In [ ]:
dev = pd.read_csv(f'{paths.RESULTS}/05_dev_results.csv')
DEV = []
for task in TASKS:
    cfg = FT[task].cfg
    d = dev[(dev.task == task) & (dev.config == cfg['model']['config']) & (dev.alpha_0 == 0.05) & (dev.beta == 0.3)]
    for _, r in d.iterrows():
        for kk in (0, 1):
            base = {'task': task, 'kind': kk, 'coverage': r[f'release_rate_class{kk}'], 'error_rate': r[f'selective_error_class{kk}'],
                    'mean_size': r[f'released_size_class{kk}'], 'median_support': np.nan, 'source': 'development (notebook 05)', 'draw': 0}
            if r['lambda'] == FT[task].lam:
                DEV.append({**base, 'factor': 'candidates M', 'level': f"M = {int(r.M)}", 'frozen': int(r.M) == FT[task].M})
            if int(r.M) == FT[task].M:
                lv = 'Occam prior only' if r['lambda'] == 1.0 else f"mixture, lambda = {r['lambda']:g}"
                DEV.append({**base, 'factor': 'prior', 'level': lv, 'frozen': r['lambda'] == FT[task].lam})
DEV = pd.DataFrame(DEV)
ALL = pd.concat([ABL, DEV], ignore_index=True)
ALL['policy_certified'] = ALL['policy_certified'].astype(float)
ALL['kind'] = ALL['kind'].map(KIND)
ALL.to_csv(f'{paths.RESULTS}/12_ablations.csv', index=False)
SUM = ALL.groupby(['task', 'source', 'factor', 'level', 'kind'], sort=False).agg(
    frozen=('frozen', 'max'), draws=('draw', 'nunique'), explained_share=('coverage', 'mean'),
    explained_lo=('coverage', lambda s: s.quantile(0.025)), explained_hi=('coverage', lambda s: s.quantile(0.975)),
    test_error=('error_rate', 'mean'), mean_size=('mean_size', 'mean'), median_support=('median_support', 'mean'),
    policy_certified=('policy_certified', 'mean'), U_sel=('U_sel', 'mean')).reset_index()
SUM.to_csv(f'{paths.RESULTS}/12_ablations_summary.csv', index=False)
for (task, kind), g in SUM.groupby(['task', 'kind'], sort=False):
    print(f'\n{NAMES[task]}, {kind} claims')
    print(g.drop(columns=['task', 'kind']).round(4).to_string(index=False))

## Figure

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False})
FACTORS = ['prior', 'explanation construction', 'candidates M', 'calibration size']
TITLE = {'prior': 'Prior', 'explanation construction': 'Explanation construction', 'candidates M': 'Number of candidate orders',
         'calibration size': 'Size of calibration split 1'}
COL = {'cdc_diabetes': ('#0072B2', 'o'), 'brfss2015_heart': ('#D55E00', 's')}
fin = SUM[SUM.source == 'final splits']
heights = [fin[fin.factor == f].level.nunique() + 1.2 for f in FACTORS]
fig, axes = plt.subplots(len(FACTORS), 2, figsize=(7.4, 8.2), sharex='col', gridspec_kw={'height_ratios': heights})
for i, fac in enumerate(FACTORS):
    levels = list(dict.fromkeys(fin[fin.factor == fac].level))
    for j, kind in enumerate(KIND.values()):
        ax = axes[i, j]
        for k, task in enumerate(TASKS):
            g = fin[(fin.factor == fac) & (fin.kind == kind) & (fin.task == task)].set_index('level').reindex(levels)
            yv = np.arange(len(levels))[::-1] + (0.15 if k == 0 else -0.15)
            col, mk = COL[task]
            ax.errorbar(g.explained_share, yv, xerr=[g.explained_share - g.explained_lo, g.explained_hi - g.explained_share],
                        fmt=mk, color=col, ms=4.5, lw=1, capsize=0, label=NAMES[task])
        frozen = set(fin[(fin.factor == fac) & fin.frozen].level)
        ax.set_yticks(np.arange(len(levels))[::-1])
        ax.set_yticklabels([l.replace('lambda', 'λ') + (' (frozen)' if l in frozen else '') for l in levels] if j == 0 else [])
        ax.set_ylim(-0.7, len(levels) - 0.3)
        ax.grid(axis='x', color='0.9', lw=0.6)
        if i == 0:
            ax.set_title(f'{kind.capitalize()} claims', fontsize=9)
        if j == 0:
            ax.text(-0.02, 1.02, TITLE[fac], transform=ax.transAxes, ha='right', va='bottom', fontsize=8.5, fontweight='bold')
for ax in axes[-1]:
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
axes[-1, 0].set_xlim(0, 0.75); axes[-1, 1].set_xlim(0, 0.45)
fig.supxlabel('Share of explained test patients given an explanation', x=0.62, fontsize=8.5)
h, l = axes[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.6, 1.0))
fig.tight_layout(rect=(0, 0, 1, 0.97))
for ext in ('png', 'pdf'):
    fig.savefig(paths.FIGURES / f'fig_ablations_frozen_model.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
key = SUM[(SUM.source == 'final splits')]
write_json(f'{paths.RESULTS}/12_ablations.json', {
    'seed': SEED, 'frozen_config_sha256': {t: FT[t].cfg_sha for t in TASKS}, 'box_table_sha256': REC08['box_table_sha256'],
    'calibration_subsamples': {'sizes': N1_SIZES, 'draws': N1_DRAWS},
    'rows': key.to_dict('records')})
print('written: results/12_ablations.csv, 12_ablations_summary.csv, 12_ablations.json, figures/fig_ablations_frozen_model')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "12: one-factor ablations on the frozen model (prior, explanation construction, selection rule, candidates, calibration size) for tasks B and A, figure"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)